# Demo

## Setup

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
model.eval()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## Konfiguration

In [14]:
print(f"vocab_size: {tokenizer.vocab_size:,}")
print(f"n_layer:    {model.config.n_layer}")
print(f"n_head:     {model.config.n_head}")
print(f"d_model:    {model.config.n_embd}")
print(f"Parameter:  {sum(p.numel() for p in model.parameters()):,}")


vocab_size: 50,257
n_layer:    12
n_head:     12
d_model:    768
Parameter:  124,439,808


## Tokenisierung

In [ ]:
text = "The capital of France is"
ids = tokenizer.encode(text)
print(f"IDs: {ids}")
for i, tid in enumerate(ids):
    print(f"  Pos {i}: ID {tid:5d} -> {tokenizer.decode([tid])!r}")

# Führendes Leerzeichen — wichtig!
print()
print(tokenizer.encode('France'))   # zerlegt in 2 Tokens
print(tokenizer.encode(' France'))  # ein Token


IDs: [3792, 6342, 262, 3139, 286, 4881, 30]
  Pos 0: ID  3792 -> 'Is'
  Pos 1: ID  6342 -> ' Paris'
  Pos 2: ID   262 -> ' the'
  Pos 3: ID  3139 -> ' capital'
  Pos 4: ID   286 -> ' of'
  Pos 5: ID  4881 -> ' France'
  Pos 6: ID    30 -> '?'

[28572]
[4881]


## Spezial tokens

In [16]:
print(f"eos_token: {tokenizer.eos_token!r} (ID {tokenizer.eos_token_id})")
print(f"pad_token: {tokenizer.pad_token!r}")  # None bei GPT-2

# Falls Batch-Encoding gebraucht wird:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


eos_token: '<|endoftext|>' (ID 50256)
pad_token: None


## Forward Pass + Logit Difference

In [17]:
inputs = tokenizer(text, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model(**inputs)

last_logits = outputs.logits[0, -1, :]
probs = torch.softmax(last_logits, dim=-1)

# Top-5
top_probs, top_ids = torch.topk(probs, k=5)
for p, tid in zip(top_probs.tolist(), top_ids.tolist()):
    print(f"  {tokenizer.decode([tid])!r:15s}  P={p:.4f}")

# Logit Difference
paris_id  = tokenizer.encode(' Paris')[0]
london_id = tokenizer.encode(' London')[0]
ld = (last_logits[paris_id] - last_logits[london_id]).item()
print(f"\nLogit Difference (Paris - London): {ld:+.3f}")


  '\n'             P=0.2887
  ' The'           P=0.0423
  ' It'            P=0.0281
  ' Or'            P=0.0212
  ' I'             P=0.0203

Logit Difference (Paris - London): +2.137


# Mini Übung

In [18]:
def logit_difference(prompt: str, correct_token: str, incorrect_token: str) -> float:
    """Berechnet LD = logit(correct) - logit(incorrect) für die letzte Position."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**inputs)
    last_logits = out.logits[0, -1, :]
    
    correct_ids   = tokenizer.encode(correct_token)
    incorrect_ids = tokenizer.encode(incorrect_token)
    assert len(correct_ids) == 1, f"{correct_token!r} ist nicht 1 Token: {correct_ids}"
    assert len(incorrect_ids) == 1, f"{incorrect_token!r} ist nicht 1 Token: {incorrect_ids}"
    
    return (last_logits[correct_ids[0]] - last_logits[incorrect_ids[0]]).item()

# Test
print(logit_difference("The capital of France is", " Paris", " London"))  # > 0
print(logit_difference("The capital of Poland is", " Paris", " Warsaw"))  # < 0


2.72283935546875
-6.351264953613281


# Übergabe an Gruppe C

### Übergabe an Gruppe C

1. **Globale Objekte bereitstellen**
   - `tokenizer` und `model` sind als **globale Variablen** initialisiert.
   - Beide nachfolgenden Methoden bauen direkt darauf auf.

2. **`logit_difference(prompt, correct, incorrect)` verwenden**
   - Die Funktion berechnet die Logit-Differenz: $LD = \text{logit}(correct) - \text{logit}(incorrect)$.
   - Dieses Signal wird für die **Logit-Lens-Auswertung pro Schicht** weiterverwendet.

3. **Prompt-Paare für das Projekt-Phänomen definieren**
   - Für die Attention-Differenz braucht ihr immer ein Paar aus:
     - **Clean Prompt** (korrekter/faktischer Kontext)
     - **Corrupted Prompt** (gezielt veränderter Kontext)
   - Nutzt in beiden Prompts möglichst identische Struktur, damit Unterschiede interpretierbar bleiben.

4. **Kritische Token-Position festhalten**
   - Die relevante Token-Position ist bereits identifiziert.
   - Diese Position bestimmt, **welche Spalte der Attention-Matrix** für die Analyse (Clean vs. Corrupted) ausgewertet wird.